In [0]:
%pip install xgboost scikit-learn pyarrow boto3 -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os, boto3
os.environ["AWS_ACCESS_KEY_ID"] = "Your Access Key "
os.environ["AWS_SECRET_ACCESS_KEY"] = "Your Secret Access Key "
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
s3 = boto3.client("s3")
BUCKET = "nbapredictions-sthomas26-ncf"
SEASON = "2024-25"

In [0]:
import pandas as pd

os.makedirs("/tmp/gold", exist_ok=True)
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"gold/games/season={SEASON}/")
for obj in resp["Contents"]:
    key = obj["Key"]
    fname = key.split("/")[-1]
    if fname.endswith(".parquet"):
        s3.download_file(BUCKET, key, f"/tmp/gold/{fname}")

df = pd.read_parquet("/tmp/gold/")
print(df.shape)
df.head()

(2757, 24)


,GAME_ID,GAME_DATE,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,MATCHUP,WL,PTS,FG_PCT,FG3_PCT,FT_PCT,REB,AST,TOV,PLUS_MINUS,HOME,WIN,PREV_GAME_DATE,REST_DAYS,AVG_PTS_L10,AVG_REB_L10,AVG_AST_L10,AVG_TOV_L10,AVG_PLUS_MINUS_L10
0,0012400011,2024-10-07,15020,NZB,New Zealand Breakers,NZB @ PHI,L,84,0.361,0.143,0.800,31,11,20,-55.0,0,0,2024-10-04,3,87.000000,35.000000,23.0,16.000000,-29.000000
1,0012400029,2024-10-10,15020,NZB,New Zealand Breakers,NZB @ OKC,L,89,0.386,0.357,1.000,38,18,13,-28.0,0,0,2024-10-07,3,85.500000,33.000000,17.0,18.000000,-42.000000
2,0012400046,2024-10-14,1610612737,ATL,Atlanta Hawks,ATL vs. PHI,L,89,0.372,0.295,0.545,52,27,20,-15.0,1,0,2024-10-08,6,131.000000,43.000000,35.0,19.000000,1.000000
3,0012400025,2024-10-16,1610612737,ATL,Atlanta Hawks,ATL @ MIA,L,111,0.425,0.378,0.769,38,28,22,-9.0,0,0,2024-10-14,2,110.000000,47.500000,31.0,19.500000,-7.000000
4,0012400064,2024-10-17,1610612737,ATL,Atlanta Hawks,ATL @ OKC,L,99,0.386,0.296,1.000,51,25,17,-5.0,0,0,2024-10-16,1,110.333333,44.333333,30.0,20.333333,-7.666667


In [0]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

FEATURES = ["HOME", "REST_DAYS", "AVG_PTS_L10", "AVG_REB_L10",
            "AVG_AST_L10", "AVG_TOV_L10", "AVG_PLUS_MINUS_L10"]
TARGET = "WIN"

X = df[FEATURES].fillna(0)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("/nba-win-predictor")

with mlflow.start_run():
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric="logloss"
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 4)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("auc", auc)
    mlflow.xgboost.log_model(model, "model")

    print(f"Accuracy: {acc:.3f}")
    print(f"AUC: {auc:.3f}")

2026/05/14 07:26:40 INFO mlflow.tracking.fluent: Experiment with name '/nba-win-predictor' does not exist. Creating a new experiment.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-e114a934-e302-4ff7-b7b2-98b3014fa571/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [07:26:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/05/14 07:27:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-da2fbc46-b04e.cloud.databricks.com/ml/experiments/1245895587867442/models/m-0c8a791223314e45a5d84a9f778cf1aa?o=7474659238273320
2026/05/14 07:27:05 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer t

Accuracy: 0.585
AUC: 0.633


In [0]:
import pickle

with open("/tmp/nba_model.pkl", "wb") as f:
    pickle.dump(model, f)

s3.upload_file("/tmp/nba_model.pkl", BUCKET, "model/nba_model.pkl")
print("Model saved to S3 ✅")

Model saved to S3 ✅
